## Imports

In [1]:
# | code-fold: true
# | code-summary: "Load packages"
# | output: false


import os
import numpy as np
import os
import numpy as np
from sympy import Matrix, sqrt, Piecewise
import sympy as sp
import pytest
from attr import define, field
from sympy import MutableDenseNDimArray as Arr


from zoomy_core.fvm.solver_numpy import Settings
from zoomy_core.model.basemodel import Model, eigenvalue_dict_to_matrix
import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
from zoomy_core.misc.misc import Zstruct, ZArray
import zoomy_core.misc.misc as misc
import zoomy_firedrake.firedrake_solver as dg
import zoomy_firedrake.firedrake_solver_animate_amr as dg_amr


In [ ]:
@define(frozen=True, slots=True, kw_only=True)
class SWE(Model):
    dimension: int = 2
    variables: Zstruct = field(init=False)
    aux_variables: Zstruct = field(default=1)
    _default_parameters: dict = field(
        init=False, factory=lambda: {"g": 9.81, "ex": 0.0, "ey": 0.0, "ez": 1.0, "rho": 1000.0, "n": 0.01, "eps":1e-4}
    )
    
    def __attrs_post_init__(self):
        object.__setattr__(self, "variables", self.dimension + 2)
        super().__attrs_post_init__()
    
    def get_primitives(self):
        dim = self.dimension
        b = self.variables[0]
        h = self.variables[1]
        hinv = 1/h
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        return b, h, U, hinv

    def flux(self):
        dim = self.dimension
        b, h, U, hinv = self.get_primitives()
        g = self.parameters.g
        I = Matrix.eye(dim)
        F = Matrix.zeros(self.variables.length(), dim)
        # F[1, :] = h * U.T
        F[1, :] = sp.Matrix(self.variables[2: 2 + dim]).T
        # F[2:, :] = h * U * U.T + g / 2 * h**2 * I
        F[2:, :] = h * U * U.T
        return ZArray(F)
    
    def nonconservative_matrix(self):
        dim = self.dimension
        b, h, U, hinv = self.get_primitives()
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        g = self.parameters.g
        N = ZArray.zeros(self.n_variables, self.n_variables, dim)
        for d in range(dim):
            N[2+d, 0, d] = g * h # g * h * grad(b)
            N[2+d, 1, d] = g * h # g * h * grad(h)
        return ZArray(N)
    
    def source(self):
        eps = 1e-4
        dim = self.dimension
        _, _, U, _ = self.get_primitives()
        hinv = self.aux_variables[0]
        # Uold = Matrix(self.aux_variables[1 : 1 + dim])
        g = self.parameters.g
        n = self.parameters.n
        abs_u = sqrt(U.dot(U) + eps)
        S = Matrix.zeros(self.n_variables, 1)
        # S[2:, 0] = -n**2 * g * hinv**(1/3) * U[:, 0] * abs_u
        S[2:, 0] = n**2 * g  * (hinv**(1/3) + eps) * U[:, 0] * abs_u
        return ZArray(S).reshape(self.n_variables,)
    
@define(frozen=True, slots=True, kw_only=True)
class NumericSWE(SWE):
    disable_differentiation: bool = False
    
    def get_primitives(self):
        dim = self.dimension
        b = self.variables[0]
        h = self.variables[1]
        hinv = self.aux_variables[0]
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        
        return b, h, U, hinv
    
    def eigenvalues(self):
        ev = super().eigenvalues()
        h = self.variables[1]
        return sp.Function('conditional')(h > self.parameters.eps, ev, ZArray.zeros(*ev.shape))
    
    
    def source(self):
        delta = self.parameters.eps  # or smaller
        h = self.variables[1]
        smooth = sp.Rational(1,2)*(1 + sp.tanh((h - self.parameters.eps)/delta))

        S = super().source()
        S2 = sp.Matrix(S)
        S2 = S2.subs({h: self.parameters.eps})
        Sreg = ZArray.zeros(*S.shape)
        for i in range(self.n_variables):
            Sreg[i] = S2[i,0]
        zeros = ZArray.zeros(*S.shape)
        return sp.Function('conditional')(h > self.parameters.eps, S, zeros)
    
    def source_jacobian_wrt_aux_variables(self):
        return ZArray.zeros(
            self.n_variables
        )
    
    def source_jacobian_wrt_variables(self):
        return ZArray.zeros(
            self.n_variables
        )
                



In [3]:
main_dir = misc.get_main_directory()
input_mesh = os.path.join(main_dir, "data", "malpasset", "geo_malpasset-small.msh")
import meshio

meshio_mesh = meshio.read(input_mesh)


In [4]:
import numpy as np

def build_vertex_permutation(fd_mesh, meshio_mesh, decimal=12):
    # get coordinates (Firedrake ordering)
    coords_fd = np.round(fd_mesh.coordinates.dat.data_ro, decimal)
    dim = fd_mesh.geometric_dimension()
    coords_fd = coords_fd[:, :dim]

    # get meshio coords (Gmsh ordering)
    coords_mio = np.round(meshio_mesh.points[:, :dim], decimal)

    # build sortable structured array
    def to_struct(coords):
        return np.array([tuple(c) for c in coords],
                        dtype=[('x', float), ('y', float)] if dim == 2
                        else [('x', float), ('y', float), ('z', float)])

    A = to_struct(coords_mio)   # shape (N,) with dtype [('x',float),('y',float)]
    B = to_struct(coords_fd)

    # sort both arrays lexicographically
    order_mio = np.argsort(A, order=A.dtype.names)
    order_fd = np.argsort(B, order=B.dtype.names)

    # the permutation mapping Firedrake → meshio is:
    perm = np.empty_like(order_fd)
    perm[order_fd] = order_mio

    return perm


# Transformation to UFL Code (Medium)

### Map from Sympy to UFL

In [ ]:


bcs = BC.BoundaryConditions(
    [
        BC.Extrapolation(tag="wall"),
        BC.Extrapolation(tag="inflow"),
        BC.Extrapolation(tag="outflow"),
    ]
)


model = NumericSWE(
    dimension=2,
    boundary_conditions=bcs,
)

settings = Settings(name="Firedrake", output=Zstruct(directory="outputs/firedrake", snapshots=100, filename='dg', clean_directory=True))


In [6]:
import ufl 
IdentityMatrix = ufl.as_tensor([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

import firedrake as fd
from zoomy_core.transformation.to_ufl import UFLRuntimeModel


@define(frozen=True, slots=True, kw_only=True)
class MySolver(dg.FiredrakeHyperbolicSolver):
    
    def set_initial_condition(self, Q, mesh, meshio_mesh):
        perm = build_vertex_permutation(mesh, meshio_mesh)
        Q.dat.data[:, 0] = meshio_mesh.point_data['B'][perm]
        Q.dat.data[:, 1] = meshio_mesh.point_data['H'][perm]
        Q.dat.data[:, 2] = (meshio_mesh.point_data['H'] * meshio_mesh.point_data['U'])[perm]
        Q.dat.data[:, 3] = (meshio_mesh.point_data['H'] * meshio_mesh.point_data['V'])[perm]
        
        
    def _setup(self, mshfile, model):
        mesh = fd.Mesh(mshfile)
        runtime_model = UFLRuntimeModel(model)


        V, Vaux, Qnp1, Qs, Qn, Qaux_np1, Qaux_s, Qaux_n = self._get_functionspaces(mesh, runtime_model)
        
        V_CG  = fd.VectorFunctionSpace(mesh, "CG", 1, dim=runtime_model.n_variables)
        Q_CG = fd.Function(V_CG)

        self.set_initial_condition(Q_CG, mesh, meshio_mesh)
        Qn = fd.project(Q_CG, V)
        Qs = fd.project(Q_CG, V)
        Qnp1 = fd.project(Q_CG, V)



        self.update_Qaux(Qn, Qaux_n)
        self.update_Qaux(Qs, Qaux_s)
        self.update_Qaux(Qnp1, Qaux_np1)
        self.update_Q(Qn, Qaux_n)
        self.update_Q(Qs, Qaux_s)
        self.update_Q(Qnp1, Qaux_np1)
        
        # Collect all boundary tags
        map_boundary_tag_to_function_index = self.get_map_boundary_tag_to_boundary_function_index(model, mshfile, mesh)
        
        return mesh, runtime_model, V, Vaux, Qn, Qs, Qnp1, Qaux_n, Qaux_s, Qaux_np1, map_boundary_tag_to_function_index 

# solver = dg.FiredrakeHyperbolicSolver(settings=settings, time_end = 10.0, CFL=0.5, IdentityMatrix=IdentityMatrix)
# solver = dg_amr.FiredrakeHyperbolicSolverAMR(settings=settings, time_end = 10.0, CFL=0.2, IdentityMatrix=IdentityMatrix)
solver = MySolver(settings=settings, time_end = 10.0, CFL=0.5, IdentityMatrix=IdentityMatrix)


In [7]:
# main_dir = misc.get_main_directory()
# path_to_mesh = os.path.join(main_dir, "meshes", "square", "mesh.msh")


In [ ]:
solver.solve(input_mesh, model)

2025-11-25 15:55:15.453 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 10, time: 0.552391, dt: 0.012029, next write at time: 0.001100
2025-11-25 15:55:30.368 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 20, time: 0.675000, dt: 0.012029, next write at time: 0.002100
2025-11-25 15:55:48.613 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 30, time: 0.796285, dt: 0.012029, next write at time: 0.003100
2025-11-25 15:56:05.792 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 40, time: 0.914147, dt: 0.011215, next write at time: 0.004100
2025-11-25 15:56:23.244 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 50, time: 1.023860, dt: 0.010384, next write at time: 0.005100
2025-11-25 15:56:40.339 | INFO     | zoomy_firedrake.firedrake_solver:solve:682 - iteration: 60, time: 1.127049, dt: 0.010127, next write at time: 0.006100
2025-11-25 15:57:17.912 | INFO     | zoomy_firedrake.firedrake_s